### Gold: `fct_service_performance` — trip punctuality fact

One row per **served stop per trip-instance**, holding the **final observed delay** and precomputed
on-time / severe / terminus / cancelled flags. This is the fact the executive KPIs slice
(see [docs/GOLD_PLAN.md](../../../../docs/GOLD_PLAN.md)): on-time performance, delay severity, hotspots,
temporal, consistency.

**Method (the bit we worked through):**
1. **Collapse** the `FULL_DATASET` snapshots → keep the **latest `feed_ts` per `(trip-instance, stop)`**
   (`row_number()` over `feed_ts DESC`). The last snapshot is the firmed-up / actual value.
2. **No delay pre-filter.** On-time vs late is classified *after* the collapse, never as a `WHERE` —
   filtering `delay > 0` first would delete the on-time rows and make punctuality un-measurable.
3. **`served` flag** — a `SKIPPED` / `NO_DATA` stop, or a stop on a cancelled trip, isn't a real arrival,
   so it's excluded from punctuality.

> Distinct from `01_gold_trip_updates.ipynb` (a change-log that keeps the compressed time-series). This
> is the collapsed *final-state* performance fact. Batch full-recompute from silver; run as a job.

In [ ]:
from pyspark.sql import functions as F, Window as W

# Catalog is passed as a job parameter (default -> ${var.catalog} per target); the default here
# keeps interactive runs working.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

SILVER = f"{CATALOG}.`02_silver`.trip_updates"
GOLD   = f"{CATALOG}.`03_gold`.fct_service_performance"

ON_TIME_SEC = 359   # <= ~6 min late still counts as on-time (V/Line publishes punctuality at 5:59)
SEVERE_SEC  = 900   # > 15 min late = severe

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.`03_gold`")

# Pre-create with explicit types + liquid clustering, so INSERT OVERWRITE (below) keeps the layout.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD} (
  service_date        DATE,
  trip_id             STRING,
  route_id            STRING,
  entity_id           STRING,
  stop_sequence       INT,
  stop_id             STRING,
  is_terminus         BOOLEAN,
  served              BOOLEAN,
  cancelled           BOOLEAN,
  start_time          STRING,
  trip_rel            STRING,
  stop_rel            STRING,
  arrival_delay_sec   INT,
  departure_delay_sec INT,
  delay_sec           INT,
  on_time             BOOLEAN,
  severe              BOOLEAN,
  arrival_time        TIMESTAMP,
  departure_time      TIMESTAMP,
  event_time          TIMESTAMP,
  final_feed_ts       TIMESTAMP,
  _gold_ts            TIMESTAMP
)
USING DELTA
CLUSTER BY (service_date, route_id)
""")

In [ ]:
silver = spark.read.table(SILVER)

# 1) COLLAPSE the FULL_DATASET snapshots -> the FINAL observed row per stop.
#    entity_id = "{trip_id}|{start_date}" identifies the trip instance; latest feed_ts wins.
#    NB: NO delay filter here — classifying on-time vs late happens after the collapse (step 4).
latest = W.partitionBy("entity_id", "stop_sequence").orderBy(F.col("feed_ts").desc())
final = (silver
    .withColumn("_rn", F.row_number().over(latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn"))

# 2) DERIVE the delay + cancellation.
#    Origin stops carry only departure_delay; mid/destination only arrival_delay -> coalesce.
final = (final
    .withColumn("delay_sec", F.coalesce("arrival_delay_sec", "departure_delay_sec"))
    .withColumn("cancelled", F.coalesce("trip_rel", F.lit("SCHEDULED")).isin("CANCELED", "CANCELLED")))

# 3) SERVED = the train actually stops here. SKIPPED / NO_DATA, or any stop on a cancelled trip, don't
#    count toward punctuality.
final = final.withColumn(
    "served",
    (F.coalesce("stop_rel", F.lit("SCHEDULED")) == F.lit("SCHEDULED")) & ~F.col("cancelled"))

# 4) TERMINUS = the last stop the train actually serves. stop_sequence is sparse, so this is a windowed
#    max over served stops per trip instance, not a fixed number.
per_trip = W.partitionBy("entity_id")
final = (final
    .withColumn("_max_served_seq", F.max(F.when(F.col("served"), F.col("stop_sequence"))).over(per_trip))
    .withColumn("is_terminus", F.col("served") & (F.col("stop_sequence") == F.col("_max_served_seq"))))

# 5) PUNCTUALITY flags — only meaningful for a served stop with an observed delay; otherwise NULL, so
#    the stop drops out of AVG(on_time) rather than counting as a false pass/fail.
measurable = F.col("served") & F.col("delay_sec").isNotNull()
fact = (final
    .withColumn("on_time", F.when(measurable, F.col("delay_sec") <= ON_TIME_SEC))
    .withColumn("severe",  F.when(measurable, F.col("delay_sec") >  SEVERE_SEC))
    .select(
        F.col("start_date").alias("service_date"),
        "trip_id", "route_id", "entity_id",
        F.col("stop_sequence").cast("int").alias("stop_sequence"),
        "stop_id", "is_terminus", "served", "cancelled",
        "start_time", "trip_rel", "stop_rel",
        F.col("arrival_delay_sec").cast("int").alias("arrival_delay_sec"),
        F.col("departure_delay_sec").cast("int").alias("departure_delay_sec"),
        F.col("delay_sec").cast("int").alias("delay_sec"),
        "on_time", "severe",
        "arrival_time", "departure_time", "event_time",
        F.col("feed_ts").alias("final_feed_ts"),
        F.current_timestamp().alias("_gold_ts"),
    ))

In [ ]:
# Full recompute from silver each run. Gold is small (one row per served stop per trip-day). INSERT
# OVERWRITE replaces the data while preserving the table's liquid clustering. Column order matches the DDL.
fact.createOrReplaceTempView("_fact_stage")
spark.sql(f"INSERT OVERWRITE TABLE {GOLD} SELECT * FROM _fact_stage")

print("gold rows:", spark.table(GOLD).count())

In [ ]:
# Q1 preview — on-time performance at destination, by route. `on_time IS NOT NULL` drops unmeasurable
# stops; is_terminus gives one row per trip (its final stop), matching V/Line's destination-based metric.
dest = spark.table(GOLD).filter("is_terminus AND on_time IS NOT NULL")

display(
    dest.groupBy("route_id").agg(
        F.count("*").alias("trips"),
        F.round(F.avg(F.col("on_time").cast("int")) * 100, 1).alias("otp_pct"),
        F.round(F.avg("delay_sec"), 0).alias("avg_delay_sec"),
        F.expr("percentile_approx(delay_sec, 0.95)").alias("p95_delay_sec"),
    ).orderBy("otp_pct")
)

#### How the six questions slice this fact

| # | Question | Query shape |
|---|---|---|
| 1 | On-time performance | `avg(on_time)` where `is_terminus` (per line via `route_id`) |
| 3 | Delay severity | `percentile_approx(delay_sec, [.5,.9,.95])`, `avg(severe)` where `served` |
| 4 | Hotspots | `avg(on_time)` grouped by `stop_id` / `route_id` (all served stops), worst first |
| 5 | Temporal | `avg(on_time)` by peak/day-type (needs `dim_date`; derive from `event_time` meanwhile) |
| 6 | Consistency | `stddev(delay_sec)` / IQR by `route_id` |

- **Grain** `(service_date, trip_id, stop_sequence)` — per-stop. Destination questions filter `is_terminus`;
  stop-level questions (Q4) use every served row.
- **Q2 (reliability/cancellations)** isn't fully answerable here — explicit cancellations are the
  `cancelled` flag, but "scheduled-but-absent" needs the `calendar`/`trips` anti-join (a separate step).
- **`route_id` is the line key** (trip updates is the single V/Line realtime feed, so there's no `feed_id`
  here). Join `02_silver.routes` on `route_id` for human line names.
- **Prediction vs actual caveat:** the final row is the latest snapshot; it's the true actual only if the
  feed published the trip through completion. If the feed drops a trip early, `delay_sec` is its last
  forecast — acceptable for this dataset, worth knowing.
- **Thresholds** (`ON_TIME_SEC=359`, `SEVERE_SEC=900`) are the only knobs; change them in cell 2 and the
  flags recompute.